# Tutorial: Structure-based virtual screening in 2 commands

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NBDsoftware/biobb_vs_workflows/blob/master/notebooks/notebook_tutorial.ipynb)

This notebook runs a complete **structure-based virtual screening** pipeline using the
**workflows** shipped in
[`biobb_vs_workflows`](https://github.com/NBDsoftware/biobb_vs_workflows) — no manual chaining of
individual building blocks.

The system is the **p38α MAP kinase** (PDB [`3HEC`](https://www.rcsb.org/structure/3HEC)) solved in
complex with the anticancer drug **Imatinib** (ligand [`STI`](https://www.rcsb.org/ligand/STI)). We
use the crystal ligand only to *locate* the binding site, then screen a small ligand library against
that pocket.

**The whole pipeline:**

| Step | Command | What it does |
|------|---------|--------------|
| 1 | `cavity_analysis` | Detect protein cavities (fpocket) and keep the one at the binding site |
| 2 | `vs_autodock`     | Dock a ligand library into that pocket (AutoDock Vina) and rank by affinity |

We then **inspect the top-ranked hit** with NGLView.

Every command supports `--help`. Note that **docking is non-deterministic** — re-running may give
slightly different scores/poses.

## 1. Set up the environment

**Running on Google Colab?** The cell below installs everything: it downloads Miniforge, clones the
repo and builds the `biobb_vs_tutorial` conda environment (the BioBB packages, AutoDock Vina,
fpocket, Open Babel, this repo's workflows and the visualization libraries). The conda solve is the
slow part — run it once and wait.

**Running locally?** Skip the cell — create the environment from `notebooks/local_environment.yml`,
launch Jupyter from it, and run this notebook **from the repository root** (so the `data/` paths
resolve).

In [ ]:
import sys, os

if 'google.colab' in sys.modules:
    # Install Miniforge + mamba
    !wget -q https://github.com/conda-forge/miniforge/releases/download/26.1.0-0/Miniforge3-26.1.0-0-Linux-x86_64.sh
    !bash Miniforge3-26.1.0-0-Linux-x86_64.sh -b -p /usr/local/miniforge

    # Clone the repo and work from inside the checkout (so data/ and notebooks/ are available)
    !git clone https://github.com/NBDsoftware/biobb_vs_workflows.git
    os.chdir('biobb_vs_workflows')

    # Build the tutorial environment
    !/usr/local/miniforge/condabin/mamba env create -f notebooks/colab_environment.yml

    # Make the env importable + put its binaries first (workflow CLIs, vina, fpocket, obabel, ...)
    sys.path.append('/usr/local/miniforge/envs/biobb_vs_tutorial/lib/python3.11/site-packages')
    os.environ['PATH'] = '/usr/local/miniforge/envs/biobb_vs_tutorial/bin:' + os.environ['PATH']

    from google.colab import output
    output.enable_custom_widget_manager()

import shutil
print('Working directory:', os.getcwd())
for cmd in ('cavity_analysis', 'vs_autodock'):
    print(f'{cmd} on PATH:', bool(shutil.which(cmd)))

## 2. Inputs

We use the crystal complex bundled in the repo at `data/complex/3HEC.pdb` (protein + Imatinib) and a
small demo ligand library `data/ligands/zinc_200_425_001_reduced.sdf`. **Swap `ligand_lib` for your
own** `.sdf` or `.smi` file to screen a different set of molecules.

Let's take a first look at the raw complex — the protein cartoon plus the co-crystallized ligand
(STI) shown as sticks.

In [ ]:
import nglview

input_pdb      = "data/complex/3HEC.pdb"                       # p38a + Imatinib (STI) crystal complex
ligand_resname = "STI"                                         # co-crystallized ligand (Imatinib)
ligand_lib     = "data/ligands/zinc_200_425_001_reduced.sdf"   # demo library (swap for your own .sdf/.smi)

for p in (input_pdb, ligand_lib):
    assert os.path.exists(p), f"{p} not found - is the working dir the repo root?"

view = nglview.show_structure_file(input_pdb)
view.add_representation('licorice', selection=ligand_resname, color='green')
view.center()
view

## 3. Locate the binding site

We don't need to know the pocket beforehand — but here we *do* have a crystal ligand, so we use it to
define the binding site: every **protein residue with an atom within 5 Å of the ligand**. This gives
an [MDAnalysis](https://docs.mdanalysis.org/stable/documentation_pages/selections.html) selection
string that we hand to `cavity_analysis` to keep only the cavity sitting on top of these residues.

In [ ]:
import MDAnalysis as mda

u = mda.Universe(input_pdb)

# Protein residues with any atom within 5 A of the co-crystallized ligand
binding_site = u.select_atoms(f"protein and byres (around 5.0 resname {ligand_resname})")
resids = sorted(set(int(r) for r in binding_site.residues.resids))

# MDAnalysis-syntax selection string reused by the cavity_analysis workflow
filtering_selection = "resid " + " ".join(str(r) for r in resids)
# NGL-syntax version, only for the visualization below
ngl_selection = " or ".join(str(r) for r in resids)

print(f"{len(resids)} binding-site residues")
print("MDAnalysis selection:", filtering_selection)

Visual check — protein (grey cartoon), the crystal ligand (green sticks) and the detected
binding-site residues (ball+stick):

In [ ]:
view = nglview.show_structure_file(input_pdb)
view.clear_representations()
view.add_representation('cartoon', selection='protein', color='lightgrey')
view.add_representation('licorice', selection=ligand_resname, color='green')
view.add_representation('ball+stick', selection=ngl_selection)
view.center(ligand_resname)
view

## 4. Cavity analysis — workflow 1

`cavity_analysis` takes the PDB, **extracts the protein** (dropping the ligand, ions and waters),
runs **fpocket** to detect cavities, filters them by score/druggability/volume, and finally keeps
only the pockets whose center of mass lies within `--distance_threshold` Å of our binding-site
residues (`--filtering_selection`).

The receptor and results land under `cavity_out/3HEC/`. (We clear any previous `cavity_out/` first so
re-runs are reproducible.)

In [ ]:
shutil.rmtree('cavity_out', ignore_errors=True)

!cavity_analysis \
    --structures_path {input_pdb} \
    --filtering_selection "{filtering_selection}" \
    --distance_threshold 8 \
    --output cavity_out

The workflow writes three ranked summaries (`summary_by_score.yml`, `summary_by_drug_score.yml`,
`summary_by_volume.yml`). Our single input structure yields one model, named after the PDB file
(`3HEC`). We pick the pocket with the **highest druggability score** among those that passed the
binding-site filter, and record the paths we'll feed to the docking step:

* `receptor`    — the protein-only structure the workflow extracted (`cavity_out/3HEC/model.pdb`)
* `pockets_zip` — the filtered pockets (`.../step7_filter_residue_com/filtered_pockets.zip`)

In [ ]:
import yaml

model = "3HEC"   # single input structure -> one model, named after the PDB file
with open("cavity_out/summary_by_drug_score.yml") as f:
    summary = yaml.safe_load(f)

assert summary and model in summary, \
    "No filtered pockets found - try a larger --distance_threshold or looser filters."

pockets = summary[model]["pockets"]
best_pocket = max(pockets, key=lambda p: summary[model][p]["druggability_score"])
pocket_num = int(best_pocket.replace("pocket", ""))

print("Filtered pockets:", pockets)
print(f"Best pocket: {best_pocket} "
      f"(druggability {summary[model][best_pocket]['druggability_score']}, "
      f"volume {summary[model][best_pocket]['volume']}) -> pocket_num = {pocket_num}")

receptor    = f"cavity_out/{model}/model.pdb"
pockets_zip = f"cavity_out/{model}/step7_filter_residue_com/filtered_pockets.zip"
assert os.path.exists(receptor),    receptor
assert os.path.exists(pockets_zip), pockets_zip

Let's visualize the chosen pocket (blue surface) on the receptor to confirm it sits at the binding
site:

In [ ]:
import zipfile
from pathlib import Path

pockets_dir = "cavity_out_pockets"
shutil.rmtree(pockets_dir, ignore_errors=True)
os.mkdir(pockets_dir)
with zipfile.ZipFile(pockets_zip) as z:
    z.extractall(pockets_dir)

pocket_pqr = str(next(Path(pockets_dir).glob(f"{best_pocket}_vert.pqr")))

view = nglview.NGLWidget()
rec = view.add_component(receptor)
rec.add_representation('cartoon', color='lightgrey')
pk = view.add_component(pocket_pqr)
pk.clear()
pk.add_surface(color='skyblue', opacity=0.5, surfaceType='av', contour=True)
view.center()
view

## 5. Virtual screening — workflow 2

`vs_autodock` prepares the receptor, builds a docking box around the selected pocket, then for each
ligand in the library: protonates/converts it, docks it with **AutoDock Vina**, and ranks all ligands
by their best binding affinity. We pass the receptor and pocket produced above.

* `--keep_poses` + `--num_top_ligands 5` — save docked poses for the 5 best ligands
* `--exhaustiveness` — sampling effort (lower = faster, less accurate); `--cpus` — cores per docking

> **Tip:** this demo library has ~50 ligands, so docking takes a few minutes on CPU. Lower
> `--exhaustiveness` (e.g. `4`) to speed it up, and swap `--ligand_lib` for your own molecules.

In [ ]:
shutil.rmtree('vs_out', ignore_errors=True)

!vs_autodock \
    --ligand_lib {ligand_lib} \
    --structure_path {receptor} \
    --input_pockets_zip {pockets_zip} \
    --pocket_num {pocket_num} \
    --keep_poses \
    --num_top_ligands 5 \
    --cpus 4 \
    --exhaustiveness 8 \
    --output vs_out

## 6. Results

The screening writes a ranking to `vs_out/scores.csv` (best affinity per ligand, most negative =
strongest predicted binder) and the saved poses to `vs_out/poses/`.

In [ ]:
import pandas as pd

scores = pd.read_csv("vs_out/scores.csv")
scores.columns = [c.strip() for c in scores.columns]   # header cells have trailing spaces
scores.head(10)

Finally, load the **top-ranked ligand's docked poses** into the receptor. The pose file bundles all
poses as separate models; they're shown together as sticks inside the pocket.

In [ ]:
top = scores.iloc[0]
pose_file = f"vs_out/poses/{top['Identifier']}_{int(top['Index'])}_poses.pdb"
print(f"Top hit: {top['Identifier']}  |  affinity {top['Affinity']} kcal/mol")
assert os.path.exists(pose_file), pose_file

view = nglview.NGLWidget()
rec = view.add_component("vs_out/receptor.pdb")
rec.add_representation('cartoon', color='lightgrey')
lig = view.add_component(pose_file)
lig.clear()
lig.add_representation('licorice')   # all docked poses of the top ligand
view.center()
view

## 7. Wrap-up

In **two workflow commands** you went from a raw PDB complex to a ranked set of docked ligands:

```
cavity_analysis  →  vs_autodock
```

To go further:

* Run either command with `--help` to see all options (clustering an MD trajectory instead of a
  single PDB, defining the pocket by residues with `--pocket_selection`, box size, exhaustiveness, ...).
* `cavity_analysis` can take a whole trajectory (`--traj_path`/`--top_path`) and analyze cavities on
  representative clustered structures — useful for ensemble docking.
* Swap `ligand_lib` for your own library and re-run step 5.